In [2]:
using LowLevelFEM, LinearAlgebra

In [3]:
openGeometry("mpc-1.geo")

#openPreProcessor()

In [4]:
mat1 = Material("body")
mat2 = Material("remote")

U = Field([mat1, mat2], type=:VectorField, dim=2, fieldName=:u, rhsName=:f)
Φ = Field([mat1, mat2], type=:ScalarField, dim=2, fieldName=:φ, rhsName=:m)

Problem("mpc-1", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :m, false)

In [5]:
Ku = ∫(SymGrad(U) ⋅ D(:PlaneStress, mat1) ⋅ SymGrad(U), Ω="body")
Ku[:,:]

10×10 SparseArrays.SparseMatrixCSC{Float64, Int64} with 64 stored entries:
  98901.1    35714.3   -60439.6   …   10989.0     2747.25   ⋅    ⋅ 
  35714.3    98901.1     2747.25      -2747.25  -60439.6    ⋅    ⋅ 
 -60439.6     2747.25   98901.1      -49450.5    35714.3    ⋅    ⋅ 
  -2747.25   10989.0   -35714.3       35714.3   -49450.5    ⋅    ⋅ 
 -49450.5   -35714.3    10989.0      -60439.6    -2747.25   ⋅    ⋅ 
 -35714.3   -49450.5    -2747.25  …    2747.25   10989.0    ⋅    ⋅ 
  10989.0    -2747.25  -49450.5       98901.1   -35714.3    ⋅    ⋅ 
   2747.25  -60439.6    35714.3      -35714.3    98901.1    ⋅    ⋅ 
       ⋅          ⋅          ⋅             ⋅          ⋅     ⋅    ⋅ 
       ⋅          ⋅          ⋅             ⋅          ⋅     ⋅    ⋅ 

In [23]:
Mu = ∫(U ⋅ mat1.ρ ⋅ U, Ω="body")
Mu[:,:]

10×10 SparseArrays.SparseMatrixCSC{Float64, Int64} with 32 stored entries:
 8.72222e-10   ⋅           4.36111e-10  …  4.36111e-10   ⋅            ⋅    ⋅ 
  ⋅           8.72222e-10   ⋅               ⋅           4.36111e-10   ⋅    ⋅ 
 4.36111e-10   ⋅           8.72222e-10     2.18056e-10   ⋅            ⋅    ⋅ 
  ⋅           4.36111e-10   ⋅               ⋅           2.18056e-10   ⋅    ⋅ 
 2.18056e-10   ⋅           4.36111e-10     4.36111e-10   ⋅            ⋅    ⋅ 
  ⋅           2.18056e-10   ⋅           …   ⋅           4.36111e-10   ⋅    ⋅ 
 4.36111e-10   ⋅           2.18056e-10     8.72222e-10   ⋅            ⋅    ⋅ 
  ⋅           4.36111e-10   ⋅               ⋅           8.72222e-10   ⋅    ⋅ 
  ⋅            ⋅            ⋅               ⋅            ⋅            ⋅    ⋅ 
  ⋅            ⋅            ⋅               ⋅            ⋅            ⋅    ⋅ 

In [6]:
mpc_u = MPC(master="remote", slave="right", field=U, ux=true, uy=true)

mpc_φ = MPC(master="remote", slave="right", field=Φ)

bc2 = BoundaryCondition("remote", field=U, ux=0, uy=0)

BoundaryCondition("remote", Problem("mpc-1", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false), Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :ux => 0))

In [7]:
R = rigidRotationMap(mpc_u, mpc_φ)
R[:,:]

10×5 SparseArrays.SparseMatrixCSC{Float64, Int64} with 2 stored entries:
  ⋅    ⋅     ⋅    ⋅    ⋅ 
  ⋅    ⋅     ⋅    ⋅    ⋅ 
  ⋅   0.5    ⋅    ⋅    ⋅ 
  ⋅    ⋅     ⋅    ⋅    ⋅ 
  ⋅    ⋅   -0.5   ⋅    ⋅ 
  ⋅    ⋅     ⋅    ⋅    ⋅ 
  ⋅    ⋅     ⋅    ⋅    ⋅ 
  ⋅    ⋅     ⋅    ⋅    ⋅ 
  ⋅    ⋅     ⋅    ⋅    ⋅ 
  ⋅    ⋅     ⋅    ⋅    ⋅ 

In [8]:
Kuφ = Ku * R
Kuφ[:,:]

10×5 SparseArrays.SparseMatrixCSC{Float64, Int64} with 16 stored entries:
  ⋅   -30219.8    24725.3    ⋅    ⋅ 
  ⋅     1373.63   17857.1    ⋅    ⋅ 
  ⋅    49450.5    -5494.51   ⋅    ⋅ 
  ⋅   -17857.1    -1373.63   ⋅    ⋅ 
  ⋅     5494.51  -49450.5    ⋅    ⋅ 
  ⋅    -1373.63  -17857.1    ⋅    ⋅ 
  ⋅   -24725.3    30219.8    ⋅    ⋅ 
  ⋅    17857.1     1373.63   ⋅    ⋅ 
  ⋅         ⋅          ⋅     ⋅    ⋅ 
  ⋅         ⋅          ⋅     ⋅    ⋅ 

In [9]:
Kφ = R' * Ku * R
Kφ[:,:]

5×5 SparseArrays.SparseMatrixCSC{Float64, Int64} with 4 stored entries:
  ⋅        ⋅         ⋅     ⋅    ⋅ 
  ⋅   24725.3   -2747.25   ⋅    ⋅ 
  ⋅   -2747.25  24725.3    ⋅    ⋅ 
  ⋅        ⋅         ⋅     ⋅    ⋅ 
  ⋅        ⋅         ⋅     ⋅    ⋅ 

In [10]:
K = SystemMatrix([Ku Kuφ; Kuφ' Kφ])
K[:,:]

15×15 SparseArrays.SparseMatrixCSC{Float64, Int64} with 100 stored entries:
  98901.1    35714.3   -60439.6   …   ⋅   -30219.8    24725.3    ⋅    ⋅ 
  35714.3    98901.1     2747.25      ⋅     1373.63   17857.1    ⋅    ⋅ 
 -60439.6     2747.25   98901.1       ⋅    49450.5    -5494.51   ⋅    ⋅ 
  -2747.25   10989.0   -35714.3       ⋅   -17857.1    -1373.63   ⋅    ⋅ 
 -49450.5   -35714.3    10989.0       ⋅     5494.51  -49450.5    ⋅    ⋅ 
 -35714.3   -49450.5    -2747.25  …   ⋅    -1373.63  -17857.1    ⋅    ⋅ 
  10989.0    -2747.25  -49450.5       ⋅   -24725.3    30219.8    ⋅    ⋅ 
   2747.25  -60439.6    35714.3       ⋅    17857.1     1373.63   ⋅    ⋅ 
       ⋅          ⋅          ⋅        ⋅         ⋅          ⋅     ⋅    ⋅ 
       ⋅          ⋅          ⋅        ⋅         ⋅          ⋅     ⋅    ⋅ 
       ⋅          ⋅          ⋅    …   ⋅         ⋅          ⋅     ⋅    ⋅ 
 -30219.8     1373.63   49450.5       ⋅    24725.3    -2747.25   ⋅    ⋅ 
  24725.3    17857.1    -5494.51      ⋅    -2747

In [11]:
bc = BoundaryCondition("left", field=U, ux=0, uy=0)

fu = ∫(U ⋅ [0, 0], Γ="remote")
fφ = ∫(Φ ⋅ 1, Γ="remote")

nodal ScalarField
[0.0; 0.0; … ; 0.0; 1.0;;]

In [12]:
DoFs(fu)

10×1 Matrix{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [13]:
DoFs(fφ)

5×1 Matrix{Float64}:
 0.0
 0.0
 0.0
 0.0
 1.0

In [14]:
F = SystemVector([fu, fφ])

SystemVector([0.0; 0.0; … ; 0.0; 1.0;;], nothing, Problem[Problem("mpc-1", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false), Problem("mpc-1", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :m, false)], [0, 10])

In [15]:
u, φ = solveField(K, F, support=[bc], mpc=[mpc_u, mpc_φ])

(VectorField(Matrix{Float64}[], [0.0; 0.0; … ; -1.338937888956732e-21; 2.022222222222221e-5;;], [0.0], Int64[], 1, :v2D, Problem("mpc-1", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false)), ScalarField(Matrix{Float64}[], [0.0; 4.044444444444443e-5; … ; 0.0; 4.044444444444443e-5;;], [0.0], Int64[], 1, :scalar, Problem("mpc-1", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8,

In [16]:
showDoFResults(u, name="u", visible=true)

0

In [17]:
DoFs(φ)

5×1 Matrix{Float64}:
 0.0
 4.044444444444443e-5
 4.044444444444443e-5
 0.0
 4.044444444444443e-5

In [18]:
u2 = u + R * φ
DoFs(u2)

10×1 Matrix{Float64}:
  0.0
  0.0
  2.0222222222222215e-5
  2.022222222222221e-5
 -2.0222222222222215e-5
  2.022222222222221e-5
  0.0
  0.0
 -1.338937888956732e-21
  2.022222222222221e-5

In [19]:
showDoFResults(u2, name="u2")

1

In [20]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
